# QA Generator

Generates ground-truth QA pairs from documents in the knowledge base.

In [6]:
import os
import sys
import json
import random
from tqdm.auto import tqdm
from openai import OpenAI
from sqlalchemy.orm import Session
from sqlalchemy import func

# Add project root to path
sys.path.append("../..")

from app.db.session import get_session_factory
from app.db.models import KnowledgeBase, Folder, Document, ChunkMetadata
from app.rag.retriever import ensure_connection

# ---- Configuration ----
# KNOWLEDGE_BASE_ID = "8825ed2f-fef5-47bb-ad46-8ed7f3caad95"
# KNOWLEDGE_BASE_ID = "5b15d3bd-b260-4502-9684-ed9113c1f223"
# KNOWLEDGE_BASE_ID = "0be5c50c-e103-493c-8e78-4fb58574be1d"
KNOWLEDGE_BASE_ID = "1a6b8438-8fec-43dd-884f-9adcfd979a79"

MODEL_NAME = "Gemma-4-31B-IT-NVFP4"
NUM_SAMPLES = 100
OUTPUT_FILE = "./results/qa_pairs.jsonl"
# GENERAL_OUTPUT_FILE = "./results/general_qa_pairs.jsonl"

LLM_BASE_URL = "http://localhost:8001/v1"


## 1. Connectivity Test

In [7]:
client = OpenAI(base_url=LLM_BASE_URL, api_key="not-needed")

try:
    models = client.models.list()
    print("vLLM is reachable. Available models:")
    for m in models.data:
        print(f" - {m.id}")
except Exception as e:
    print(f"Error connecting to vLLM: {e}")

vLLM is reachable. Available models:
 - Gemma-4-31B-IT-NVFP4


## 2. Load Documents from Database

In [8]:
SessionLocal = get_session_factory()
db = SessionLocal()

# Ensure Milvus connection is alive (even if we sample from SQLite, good to test)
ensure_connection()

# 1. Find the KB
kb = db.query(KnowledgeBase).filter(KnowledgeBase.id == KNOWLEDGE_BASE_ID).first()
if not kb:
    print(f"Knowledge Base {KNOWLEDGE_BASE_ID} not found.")
else:
    print(f"Found KB: {kb.name} (Collection: {kb.collection_name})")

# 2. Find all documents in this KB
docs = (
    db.query(Document)
    .join(Folder, Document.folder_id == Folder.id)
    .filter(Folder.knowledge_base_id == KNOWLEDGE_BASE_ID)
    .all()
)
doc_ids = [d.id for d in docs]
print(f"Found {len(docs)} documents in KB.")

# 3. Sample chunks from these documents
sampled_docs = (
    db.query(Document)
    .join(Folder, Document.folder_id == Folder.id)
    .filter(Folder.knowledge_base_id == KNOWLEDGE_BASE_ID)
    .order_by(func.random())
    .limit(NUM_SAMPLES)
    .all()
)
print(f"Sampled {len(sampled_docs)} unique documents.")


Found KB: Eval (Collection: kb_eval_1a6b8438)
Found 383 documents in KB.
Sampled 100 unique documents.


## 3. Generate QA Pairs

In [9]:
def get_chunk_window(db_session, target_chunk, window_size=1) -> str:
    """
    Retrieve the target chunk along with its surrounding chunks for more context.
    window_size=1 means we grab [chunk-1, chunk, chunk+1].
    """
    chunks = (
        db_session.query(ChunkMetadata)
        .filter(ChunkMetadata.document_id == target_chunk.document_id)
        .filter(ChunkMetadata.chunk_index >= target_chunk.chunk_index - window_size)
        .filter(ChunkMetadata.chunk_index <= target_chunk.chunk_index + window_size)
        .order_by(ChunkMetadata.chunk_index)
        .all()
    )
    return "\n\n".join([c.content for c in chunks])


def generate_qa_pair(text: str) -> dict:
    """Generate a single QA pair from the provided text using vLLM."""
    # prompt = (
    # prompt = (
    prompt = (
    prompt = (
        "You are an expert dataset engineer building a retrieval-augmented generation (RAG) benchmark.\n"
        "Given the text snippet below, generate exactly ONE question and answer pair.\n\n"
        "=== LANGUAGE RULE ===\n"
        "Generate the question and answer in the SAME LANGUAGE as the text snippet. "
        "If the snippet is primarily English -> output EN. If Indonesian -> output ID. If Mandarin -> output ZH. "
        "Do not translate; preserve the original linguistic context.\n\n"
        "=== ABSOLUTE CONSTRAINTS (STANDALONE TEST) ===\n"
        "Before outputting, verify that a user who has NEVER seen the text could still understand the question. "
        "If the question contains ANY hidden pointer to the source text, it FAILS.\n\n"
        "FORBIDDEN PATTERNS (automatic failure):\n"
        "  • Sermon/Lecture Codes & Numbers: DO NOT include any specific sermon codes, lecture numbers, or document codes in the question "
        "    (e.g., 'GRII 1805', 'MRI-1805', 'MRIS-400', '1805', '400', '1742', 'GRII SORE 291', 'GRII SORE 400', 'GRII 1811'). "
        "    These are too ideal/specific and violate a standalone query. Instead, use the name of the speaker and the actual topic.\n"
        "  • English: 'according to the text/passage/document', 'based on the above', 'the aforementioned', 'that/this/these/those [noun]', 'the former/latter', 'it/they/them' (when antecedent is unnamed), 'what was discussed', 'the thing mentioned'.\n"
        "  • Indonesian: 'menurut teks/dokumen', 'dalam tulisan ini', 'tersebut', 'itu/ini' (as back-reference), 'yang dimaksud/disebutkan', 'hal tersebut', 'topik yang dibicarakan', 'doa tersebut'.\n"
        "  • Mandarin: '根据文本/文章', '上述/前述', '该/此/那' (as back-reference), '提到的', '文中所说的', '讨论的内容', '那个祈祷/观点'.\n\n"
        "=== ENTITY SUBSTITUTION RULE ===\n"
        "Every pronoun or vague nominal phrase MUST be replaced by the full, specific noun phrase from the text. "
        "Think of it as writing a search engine query: the question itself must contain the keywords that would lead someone to this document.\n"
        "  BAD:  'What did he ask for in that prayer?'\n"
        "  GOOD: 'What blessing did Pastor Stephen Tong ask for in the opening prayer for the congregation\'s understanding of faith?'\n\n"
        "=== RETRIEVAL OPTIMIZATION ===\n"
        "The question must be dense with specific entities (proper nouns, dates, events, concepts, actions) so that a vector embedding model can match it to the correct chunk without any surrounding context.\n\n"
        "=== OUTPUT FORMAT ===\n"
        "Return ONLY a raw JSON object. No markdown code fences, no explanations, no preamble.\n"
        '{"question": "...", "answer": "..."}\n\n'
        f"TEXT SNIPPET:\n{text}\n"
    )
    for attempt in range(3):
        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.7,
                max_tokens=512,
            )
            content = response.choices[0].message.content.strip()
            if content.startswith("```json"):
                content = content[7:-3].strip()
            elif content.startswith("```"):
                content = content[3:-3].strip()
                
            qa = json.loads(content)
            if "question" in qa and "answer" in qa:
                return qa
        except Exception as e:
            print(f"Attempt {attempt+1} failed: {e}")
    
    return None

# def generate_general_qa_pair(text: str) -> dict:
#     """Generate a single QA pair from the provided text using vLLM."""
#     for attempt in range(3):
#         try:
#             response = client.chat.completions.create(
#                 model=MODEL_NAME,
#                 messages=[{"role": "user", "content": prompt}],
#                 temperature=0.7,
#                 max_tokens=512,
#             )
#             content = response.choices[0].message.content.strip()
#             if content.startswith("```json"):
#                 content = content[7:-3].strip()
#             elif content.startswith("```"):
#                 content = content[3:-3].strip()
                
#             qa = json.loads(content)
#             if "question" in qa and "answer" in qa:
#                 return qa
#         except Exception as e:
#             print(f"Attempt {attempt+1} failed: {e}")
    
#     return None



In [10]:
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    pass
count = 0
# Open in append mode for streaming
with open(OUTPUT_FILE, "a", encoding="utf-8") as f:
    for doc in tqdm(sampled_docs, desc="Generating QA pairs"):
        # Get one representative chunk from this document
        chunk = db.query(ChunkMetadata).filter(ChunkMetadata.document_id == doc.id).first()
        if not chunk: continue
        
        # Get surrounding context (sliding window)
        context_text = get_chunk_window(db, chunk, window_size=1)
        
        qa = generate_qa_pair(context_text)
        if qa:
            new_pair = {
                "document_source": doc.title,
                "doc_id": doc.id,
                "chunk_id": chunk.id,
                "question": qa["question"],
                "answer": qa["answer"]
            }
            # Save immediately
            f.write(json.dumps(new_pair, ensure_ascii=False) + "\n")
            f.flush()
            count += 1
print(f"Successfully generated and saved {count} unique QA pairs to {OUTPUT_FILE}")

Generating QA pairs:   0%|          | 0/100 [00:00<?, ?it/s]

Successfully generated and saved 100 unique QA pairs to ./results/qa_pairs.jsonl
